### Harry potter rag 

In [1]:
from dotenv import load_dotenv
import os
import pandas as pd

load_dotenv()
api_key = os.getenv("GEMINI_API_KEY") # This is the API key for the Gemini API, which is used to access the language model.

csv_path = "data/harry_potter_cleaned.csv" 

df = pd.read_csv(csv_path)
df.head() 

,name,gender,job,house,patronus,species,blood_status,loyalty,skills,birth,...,titles,category,hand,light,difficulty,inventors,ingredients,side_effects,time,title
0,Albus Percival Wulfric Brian Dumbledore,Male,Headmaster,Gryffindor,Phoenix,Human,Half-blood,Dumbledore's Army | Order of the Phoenix | Hog...,Considered by many to be one of the most power...,Late August 1881,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Quirinus Quirrell,Male,Defence Against the Dark Arts(1991-1992),Ravenclaw,Non-corporeal,Human,Half-blood,Lord Voldemort,Learned in the theory of Defensive Magic| less...,"26 September,1970 or earlier",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Fred Weasley,Male,Student,Gryffindor,Unknown,Human,Pure-blood,Dumbledore's Army | Order of the Phoenix | Hog...,Beater,"1 April, 1978",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Nymphadora Tonks,Female,Auror,Hufflepuff,"Jack rabbit, Wolf",Human,Half-blood,Ministry of Magic | Order of the Phoenix,"Talented Auror, Metamorphmagus",1 September 1972- 31 August 1973,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Lavender Brown,Female,Student,Gryffindor,Unknown,Human,Pure-blood,Dumbledore's Army |Hogwarts School of Witchcra...,Divination,1 September 1979- 31 August 1980,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [2]:
# Load the CSV file using the CSVLoader from langchain_community
from langchain_community.document_loaders.csv_loader import CSVLoader

loader = CSVLoader(file_path=csv_path)
documents = loader.load()
print(f"Number of documents: {len(documents)}")
print(f"First document: {documents[0]}")

Number of documents: 5766
First document: page_content='name: Albus Percival Wulfric Brian Dumbledore
gender: Male
job: Headmaster
house: Gryffindor
patronus: Phoenix
species: Human
blood_status: Half-blood
loyalty: Dumbledore's Army | Order of the Phoenix | Hogwarts School of Witchcraft and Wizardry
skills: Considered by many to be one of the most powerful wizards of his time
birth: Late August 1881
death: 30 June, 1997
source: characters
incantation: 
type: 
effect: 
characteristics: 
alias_names: 
titles: 
category: 
hand: 
light: 
difficulty: 
inventors: 
ingredients: 
side_effects: 
time: 
title: ' metadata={'source': 'data/harry_potter_cleaned.csv', 'row': 0}


In [3]:
# Creating a embeddings model using the GoogleGenerativeAIEmbeddings class from langchain_google_genai
from langchain_google_genai import GoogleGenerativeAIEmbeddings
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    google_api_key=api_key
)


In [4]:
# Create a vector store 
from langchain_chroma import Chroma
vector_store = Chroma(
    collection_name="harry_potter",                  # name of the collection
    embedding_function=embeddings,              # give the embedding model as an argument
    persist_directory="./chroma_harry_potter_db",    # where to save data locally
)

In [5]:
# Add a smaller sample first before adding all documents, to make sure everything works as expected. 
# And to avoid hitting rate limits with the embedding model.
sample_documents = documents[:100]

document_ids = vector_store.add_documents(documents=sample_documents)

print(len(document_ids))

100


In [6]:
# Checking the similarity search with a sample question.
question = "What house does Harry Potter belong to?"

results = vector_store.similarity_search(question, k=3)

for result in results:
    print(result.page_content)
    print("---")

name: Harry James Potter
gender: Male
job: Student
house: Gryffindor
patronus: Stag
species: Human
blood_status: Half-blood
loyalty: Albus Dumbledore | Dumbledore's Army | Order of the Phoenix | Hogwarts School of Witchcraft and Wizardry
skills: Parseltongue| Defence Against the Dark Arts | Seeker
birth: 31 July 1980
death: 
source: characters
incantation: 
type: 
effect: 
characteristics: 
alias_names: 
titles: 
category: 
hand: 
light: 
difficulty: 
inventors: 
ingredients: 
side_effects: 
time: 
title: 
---
name: James Potter
gender: Male
job: 
house: Gryffindor
patronus: Stag
species: Human
blood_status: Pure-blood
loyalty: Order of the Phoenix
skills: Animagus| Seeker
birth: 27 March, 1960
death: 31 October, 1981
source: characters
incantation: 
type: 
effect: 
characteristics: 
alias_names: 
titles: 
category: 
hand: 
light: 
difficulty: 
inventors: 
ingredients: 
side_effects: 
time: 
title: 
---
name: Hermione Jean Granger
gender: Female
job: Student
house: Gryffindor
patronus:

In [7]:
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(
    model="gemini-3-flash-preview",
    google_api_key=api_key
)

In [8]:
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

In [9]:
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

template = """
You are a helpful assistant that answers questions about a Harry Potter dataset.
Use the retrieved data to answer the question.
If the retrieved data does not contain the information needed to answer, say you don't know.

Question: {question}

Retrieved data:
{retrieved_data}
"""


In [10]:
prompt = PromptTemplate.from_template(template)
parser = StrOutputParser()

In [11]:
chain = (
    {
        "retrieved_data": retriever,
        "question": RunnablePassthrough()
    }
    | prompt
    | model
    | parser
)

In [18]:
question = input("Ask a question about the Harry Potter dataset: ")


In [19]:
answer = chain.invoke(question)

In [20]:
print(answer)

Based on the retrieved data, Ron (Ronald Bilius Weasley) is a wizard with the following characteristics:

*   **Full Name:** Ronald Bilius Weasley
*   **House:** Gryffindor
*   **Job:** Student
*   **Species:** Human
*   **Blood Status:** Pure-blood
*   **Birth Date:** 1 March 1980
*   **Patronus:** Jack Russell terrier
*   **Skills:** Wizard chess and Quidditch goalkeeping
*   **Loyalty:** Dumbledore's Army, Order of the Phoenix, and Hogwarts School of Witchcraft and Wizardry
